# 🔍 Basic RAG Pipeline — LangChain + Ollama + ChromaDB

---

## 🧠 What is RAG?

**Retrieval-Augmented Generation** gives a model facts it was never trained on. Instead of asking the model to remember your company policy, you **find the relevant text first**, then **hand it to the model** along with the question.

This notebook builds the simplest working version, step by step, with a hand-written loop. Its companion, `basic_rag_langgraph.ipynb`, turns the querying half into a LangGraph `StateGraph`.

```
┌──────────────────────────────────────────────────────────┐
│                     INDEXING PHASE                        │
│  Text File → Loader → Splitter → Embeddings → Chroma DB   │
└──────────────────────────────────────────────────────────┘
              ↓  (query time)
┌──────────────────────────────────────────────────────────┐
│                     QUERYING PHASE                        │
│  Question → Retriever (top 2 chunks) → Prompt → LLM → Answer │
└──────────────────────────────────────────────────────────┘
```

## 📦 Prerequisites & Installation

1. **Ollama installed and running** → https://ollama.com
2. **Models pulled:**
   ```bash
   ollama pull nomic-embed-text    # embedding model
   ollama pull gpt-oss:120b-cloud  # chat model
   ```
3. **Python packages** (already installed if you ran `uv sync`):
   ```bash
   pip install langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb
   ```
4. **A sample document at** `docs/company_policy.txt` (already present in this folder).

---
## 📚 Step 0 — Imports

| Import | Role |
|---|---|
| `TextLoader` | Reads a `.txt` file from disk |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `OllamaEmbeddings` | Converts text → dense vectors |
| `Chroma` | Vector database — stores & searches embeddings |
| `ChatOllama` | LangChain wrapper around the Ollama chat model |

In [ ]:
# ── Document Ingestion ─────────────────────────────────────────────
from langchain_community.document_loaders.text import TextLoader

# ── Chunking ───────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embeddings & LLM (local, via Ollama) ───────────────────────────
from langchain_ollama import ChatOllama, OllamaEmbeddings

# ── Vector Store (local, persisted to disk) ────────────────────────
from langchain_chroma import Chroma

print("✅ All imports successful")

---
## 📄 Step 1 — Load the Document

`TextLoader` reads the raw `.txt` file and wraps it in a `Document` object with two fields: `page_content` (the text) and `metadata` (where it came from).

In [ ]:
print("Loading document...\n")

loader = TextLoader(
    "docs/company_policy.txt",
    encoding="utf-8"
)

documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview (first 300 chars):")
print(documents[0].page_content[:300])
print(f"\n🏷️  Metadata: {documents[0].metadata}")

---
## ✂️ Step 2 — Split Document into Chunks

Models and vector searches work best on small pieces. We cut the text into chunks of about 200 characters, with 50 characters of **overlap** so a sentence cut at a boundary still appears whole in at least one chunk.

In [ ]:
print("Splitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks Created: {len(chunks)}")
print(f"\n--- Previewing first 3 chunks ---")

for index, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {index + 1} ({len(chunk.page_content)} chars)")
    print(chunk.page_content)
    print("-" * 50)

---
## 🔢 Step 3 — Create the Embedding Model

An **embedding** turns text into a list of numbers so that texts with similar meaning end up close together. We use `nomic-embed-text`, running locally through Ollama.

In [ ]:
print("Loading Embedding Model...\n")

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("✅ Embedding Model Ready")

test_vector = embeddings.embed_query("What is the leave policy?")
print(f"\n🔢 Vector dimension: {len(test_vector)}")
print(f"📊 First 5 values: {[round(v, 4) for v in test_vector[:5]]}")

---
## 🗄️ Step 4 — Create & Persist the Vector Store (ChromaDB)

`Chroma.from_documents` embeds every chunk and stores the vectors on disk in `./chroma_db`, so you do not have to re-embed next time.

In [ ]:
print("Creating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✅ Vector Store Created & Persisted to ./chroma_db")
print(f"📦 Total vectors stored: {vectorstore._collection.count()}")

---
## 🔎 Step 5 — Create the Retriever

A retriever embeds the question and returns the `k` most similar chunks. Here `k=2`, so each question gets the 2 best-matching chunks.

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("✅ Retriever Ready")

test_query = "What is the leave policy?"
test_results = retriever.invoke(test_query)

print(f"\n🔍 Test Query: '{test_query}'")
print(f"📄 Retrieved {len(test_results)} chunk(s):")
for i, doc in enumerate(test_results):
    print(f"\n  Chunk {i+1}: {doc.page_content[:150]}...")

---
## 🤖 Step 6 — Create the Chat Model

`ChatOllama` is LangChain's wrapper around the Ollama chat model. `temperature=0` keeps answers consistent from run to run.

In [ ]:
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

print("✅ LLM Ready")

---
## 💬 Step 7 — Single-Shot Query (retrieve → prompt → generate)

Four steps, written out by hand:

1. **Retrieve** the top chunks for the question
2. **Join** them into one context string
3. **Build a prompt** that tells the model to answer *only* from that context
4. **Call the model**

The instruction "answer ONLY using the context" is what stops the model from guessing.

In [ ]:
question = "What is the work from home policy?"

# 1. Retrieve
retrieved_docs = retriever.invoke(question)

# 2. Join chunks into one context string
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

# 3. Build the grounded prompt
prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

# 4. Generate
response = llm.invoke(prompt)

print(f"❓ Question: {question}")
print(f"\n📚 Retrieved Context:\n{context}")
print(f"\n🤖 Answer: {response.content}")

---
## 🔁 Step 8 — Interactive Q&A Loop

The same four steps inside a `while True` loop. Try a question that is not in the document, such as "What is the firewall policy?" The model is still called, and it is the prompt that makes it say "not found".

In [ ]:
print("RAG Q&A System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:
    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)

    print("\n===== 📚 RETRIEVED CONTEXT =====")
    print(context)

    print("\n===== 🤖 ANSWER =====")
    print(response.content)

---
## 🧪 Bonus — Load Existing Vector Store (Skip Re-indexing)

Once the store is saved in `./chroma_db`, you can reload it directly instead of loading, splitting and embedding again.

In [ ]:
vectorstore_loaded = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

retriever = vectorstore_loaded.as_retriever(search_kwargs={"k": 2})

print(f"✅ Loaded existing store from disk")
print(f"📦 Vectors in store: {vectorstore_loaded._collection.count()}")

docs = retriever.invoke("How many days notice period is required?")
print(f"\n📄 Top chunk: {docs[0].page_content}")

---
## 🎯 Key Takeaways

| Stage | Component | Purpose |
|---|---|---|
| Load | `TextLoader` | Read the file into a `Document` |
| Split | `RecursiveCharacterTextSplitter` | Cut into small overlapping chunks |
| Embed | `OllamaEmbeddings` | Turn text into vectors |
| Store | `Chroma` | Save and search the vectors |
| Retrieve | `as_retriever(k=2)` | Find the 2 most relevant chunks |
| Generate | `ChatOllama` + grounded prompt | Answer only from the retrieved context |

## 🚀 Next

Open `basic_rag_langgraph.ipynb`: the indexing is identical, but the querying loop becomes a `StateGraph` with typed state, named nodes, and a conditional edge that skips the model call when nothing is retrieved.